In [7]:
import pandas as pd

In [8]:
%load_ext autoreload
%autoreload 2

from python_files.handlers import ZupReportProcessor as ZUPpr
from python_files.handlers import T51

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
proc = ZUPpr('Данные ЗУП.xlsx')
zup_df = proc.run()

In [10]:
zup_df

,Период,Сотрудник сокр,Подразделение,Должность,Вид_начисления,Начислено
0,2025-09-01,Кова Е. Р.,Финансовый отдел,Юрист,Оплата по окладу,115000.00
1,2025-09-01,Костюк А. В.,Финансовый отдел,Юрисконсульт,Оплата по окладу,80500.00
2,2025-09-01,Самойлова Ю. В.,Финансовый отдел,Бухгалтер,Оплата по окладу,138000.00
3,2025-09-01,Серая М. А.,Финансовый отдел,Главный бухгалтер,Оплата по окладу,169272.73
4,2025-09-01,Серая М. А.,Финансовый отдел,Главный бухгалтер,Отпуск основной,225024.75
...,...,...,...,...,...,...
811,2026-02-01,Расцветаев Н. В.,"ОП Краснодар Березанская ООО ""Винтео""",Младший программист клиент-серверных приложений,Больничный за счет работодателя,5926.38
812,2026-02-01,Сейлер К. С.,"ОП Краснодар Березанская ООО ""Винтео""",Инженер по автоматизации тестирования,Оплата по окладу,138000.00
813,2026-02-01,Сейлер К. С.,"ОП Краснодар Березанская ООО ""Винтео""",Инженер по автоматизации тестирования,Больничный за счет работодателя,10962.27
814,2026-02-01,Чеботарев А. Е.,"ОП Краснодар Березанская ООО ""Винтео""",Старший веб-программист,Оплата по окладу,317000.00


In [ ]:
proc = T51('Данные Т51.xlsx')
df_T51 = proc.run()

In [17]:
df_T51[['Сотрудник сокр','Период','Вид_начисления', 'Начислено']]

,Сотрудник сокр,Период,Вид_начисления,Начислено
0,Карукес С.,2025-09-01,Больничный за счет работодателя,2648.76
1,Морозов С. В.,2025-09-01,Больничный за счет работодателя,17021.91
2,Винниченко А. В.,2025-09-01,Командировка,35769.78
3,Ролдугин Д. В.,2025-09-01,Командировка,62407.86
4,Ткач Д. Г.,2025-09-01,Компенсация не использованных дней отдыха,6521.74
...,...,...,...,...
859,Яковлев Т. И.,2026-02-01,Отпуск основной,89626.74
860,Гаджиева А. Л.,2026-02-01,Районный коэффициент,38500.00
861,Гаджиева А. Л.,2026-02-01,Северная надбавка,10150.00
862,Винниченко А. В.,2026-02-01,Суточные сверх нормы,5800.00


In [36]:
df_pivot_zup = pd.pivot_table(zup_df, index=['Сотрудник сокр','Период', 'Вид_начисления'], values='Начислено', aggfunc='sum')
df_pivot_t51 = pd.pivot_table(df_T51, index=['Сотрудник сокр','Период', 'Вид_начисления'], values='Начислено', aggfunc='sum')

In [38]:
df_pivot_zup.to_clipboard()

In [39]:
df_pivot_t51.to_clipboard()

In [26]:
# 2. Объединяем их (аналог сводной таблицы со сравнением)
# how='outer' позволяет увидеть строки, которые есть только в одном из отчетов
df_comparison = pd.merge(
    zup_df, 
    df_T51, 
    on=['Период','Сотрудник сокр', 'Вид_начисления'], 
    how='outer', 
    suffixes=('_zup', '_t51')
)

In [27]:
# 3. Заполняем пустоты нулями (если начисления не было в одном из месяцев)
df_comparison['Начислено_zup'] = df_comparison['Начислено_zup'].fillna(0)
df_comparison['Начислено_t51'] = df_comparison['Начислено_t51'].fillna(0)

In [28]:
# 4. Считаем разницу
df_comparison['Разница'] = df_comparison['Начислено_zup'] - df_comparison['Начислено_t51']

In [29]:
# 5. Оставляем только те строки, где есть расхождения
df_diff = df_comparison[df_comparison['Разница'] != 0].copy()

In [ ]:
df_diff

In [35]:
# Применяем оформление и сохраняем
df_diff.to_excel('Сверка_ЗУП.xlsx', index=False)